# Claude API — Request Lifecycle

---

## Overview

Every interaction with Claude follows **five distinct phases**:
```
User (Client)
    |
    | 1. Request to your server
    v
Your Server
    |
    | 2. Request to Anthropic API  (with secret API key)
    v
Anthropic API
    |
    | 3. Model processing
    v
Anthropic API
    |
    | 4. Response to your server
    v
Your Server
    |
    | 5. Response to client
    v
User (Client)
```

---

## Why You Need a Server (Never Call the API from the Client)

| Concern | Detail |
|---|---|
| **Security** | API key must never be exposed in client-side code |
| **Unauthorized use** | Anyone can extract a key from browser/app code |
| **Solution** | Client talks to your server; your server talks to Anthropic |

---

## Making API Requests

Anthropic provides official SDKs for: **Python, TypeScript, JavaScript, Go, Ruby**  
You can also use plain HTTP requests.

### Required Fields in Every Request

| Field | Purpose |
|---|---|
| `api_key` | Authenticates your request to Anthropic |
| `model` | Which Claude model to use (e.g. `claude-3-sonnet`) |
| `messages` | List containing the user's input |
| `max_tokens` | Upper limit on tokens Claude can generate |

---

## Inside Claude's Processing (4 Stages)

### 1. Tokenization
- Input text is broken into **tokens** — words, subwords, spaces, or symbols
- Rule of thumb: ~1 word ≈ 1 token

### 2. Embedding
- Each token is converted into an **embedding** — a long list of numbers encoding all
  possible meanings of that token
- Captures semantic relationships between words

**Example — "quantum" could mean:**
- A discrete unit of physical quantity (physics)
- Quantum mechanics / quantum physics
- Something subatomic or extremely small
- Quantum computing

### 3. Contextualization
- Embeddings are **refined based on surrounding words** to determine the most likely meaning
- Self-attention adjusts numerical representations to highlight the appropriate definition
- This is where the Transformer architecture does its core work

### 4. Generation
- Contextualized embeddings pass through an output layer that calculates **probability
  distributions** over all possible next tokens
- Claude does **not** always pick the highest probability token — it uses controlled
  randomness to produce natural, varied responses
- After each token is selected, it is added to the sequence and the process repeats
```
[Contextualized Embeddings]
         |
         v
[Output Layer — probability over vocab]
         |
         v
[Sample next token]
         |
         v
[Append to sequence → repeat]
```

---

## When Claude Stops Generating

After each token, Claude checks three stopping conditions:

| Condition | Description |
|---|---|
| **Max tokens reached** | Hit the `max_tokens` limit you specified |
| **Natural ending** | Model generated an end-of-sequence token |
| **Stop sequence** | Encountered a predefined stop phrase |

---

## The API Response Object

The response sent back contains:

| Field | Content |
|---|---|
| `message` | The generated text |
| `usage` | Count of input tokens and output tokens |
| `stop_reason` | Why generation ended (max tokens / end token / stop sequence) |

---

## Key Takeaways

- Always route API calls through **your own server** — never the client
- Set **`max_tokens`** deliberately based on your use case
- Handle all three **stop reasons** in your application logic
- Issues can be traced to a specific phase — useful for debugging

# Making Your First Anthropic API Request (Python)

---

## 1. Environment Setup

Install required packages:
```python
%pip install anthropic python-dotenv
```

Create a `.env` file in the same directory as your notebook:
```
ANTHROPIC_API_KEY="your-api-key-here"
```

> Always add `.env` to your `.gitignore` — never commit API keys to version control.

---

## 2. Load Keys and Create Client
```python
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"
```

- `load_dotenv()` reads the `.env` file and injects the key as an environment variable
- `Anthropic()` automatically picks up `ANTHROPIC_API_KEY` from the environment

---

## 3. The `client.messages.create()` Function

The core function for all API requests. Three required parameters:

| Parameter | Type | Description |
|---|---|---|
| `model` | `str` | Which Claude model to use |
| `max_tokens` | `int` | **Safety ceiling** on response length — not a target |
| `messages` | `list` | The conversation history sent to Claude |

**Note on `max_tokens`:** Claude writes what it thinks is appropriate and stops naturally.
`max_tokens` only kicks in as a hard cutoff — it does not pad or target a length.

---

## 4. Message Structure

A conversation is a list of message dictionaries, each with two fields:
```python
{
    "role": "user",       # either "user" or "assistant"
    "content": "..."      # the actual text
}
```

| Role | Written by |
|---|---|
| `user` | You / the human |
| `assistant` | Claude's responses |

---

## 5. Making a Request
```python
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence."
        }
    ]
)
```

---

## 6. Extracting the Response
```python
message.content[0].text
```

The full response object contains metadata (token usage, stop reason, etc.),
but `.content[0].text` gives you the clean generated text directly.

---

## Full Working Example
```python
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence."
        }
    ]
)

print(message.content[0].text)
```

---

## Key Takeaways

- Never hardcode API keys — always use `.env` + `python-dotenv`
- `max_tokens` is a **safety limit**, not a length target
- The `messages` list is where conversation history lives
- Response text is at `message.content[0].text`

In [13]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"


In [14]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [15]:
print(message)

print(message.content[0].text)

Message(id='msg_01BqANynx6E7xm2uoJdQAkTZ', container=None, content=[TextBlock(citations=None, text='Quantum computing is a revolutionary computing paradigm that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster than classical computers.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=43, server_tool_use=None, service_tier='standard'))
Quantum computing is a revolutionary computing paradigm that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster tha